<a href="https://colab.research.google.com/github/keyvanmahmoudi/Data_Science_Course/blob/main/solutions/Session07_sql_for_data_analysis_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۷: SQL کاربردی برای تحلیل داده

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session07_sql_for_data_analysis.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

کتابخانه‌ها و داده‌هایی که پاسخ‌ها به آن‌ها نیاز دارند:

</div>

In [1]:
import sqlite3
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Data_Science_Course/main/data/"
db_path = Path("chinook.sqlite")
if not db_path.exists():
    urllib.request.urlretrieve(DATA_URL + "chinook.sqlite", db_path)
conn = sqlite3.connect(db_path)

def sql(query, params=None):
    return pd.read_sql_query(query, conn, params=params)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

۵ آلبومی که بیشترین تعداد آهنگ را دارند، همراه با نام هنرمندشان پیدا کنید.

</div>

In [2]:
sql("""
SELECT al.Title AS album, ar.Name AS artist, COUNT(t.TrackId) AS tracks
FROM Album  AS al
JOIN Artist AS ar ON al.ArtistId = ar.ArtistId
JOIN Track  AS t  ON t.AlbumId   = al.AlbumId
GROUP BY al.AlbumId
ORDER BY tracks DESC
LIMIT 5
""")

,album,artist,tracks
0,Greatest Hits,Lenny Kravitz,57
1,Minha Historia,Chico Buarque,34
2,Unplugged,Eric Clapton,30
3,"Lost, Season 3",Lost,26
4,"Lost, Season 1",Lost,25


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

هر کارمند پشتیبانی (`SupportRepId` در جدول `Customer`) مسئول چند مشتری است و مجموع فروش مشتریانش چقدر است؟

</div>

In [3]:
sql("""
SELECT e.FirstName || ' ' || e.LastName AS support_rep,
       COUNT(DISTINCT c.CustomerId)     AS customers,
       ROUND(SUM(i.Total), 2)           AS sales
FROM Employee AS e
JOIN Customer AS c ON c.SupportRepId = e.EmployeeId
JOIN Invoice  AS i ON i.CustomerId   = c.CustomerId
GROUP BY e.EmployeeId
ORDER BY sales DESC
""")

,support_rep,customers,sales
0,Jane Peacock,21,833.04
1,Margaret Park,20,775.40
2,Steve Johnson,18,720.16


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

میانگین طول آهنگ (به دقیقه) در هر سبک را حساب کنید و فقط سبک‌هایی با بیش از ۵۰ آهنگ را نشان دهید.

</div>

In [4]:
sql("""
SELECT g.Name AS genre,
       COUNT(*) AS tracks,
       ROUND(AVG(t.Milliseconds) / 60000.0, 2) AS avg_minutes
FROM Track AS t
JOIN Genre AS g ON t.GenreId = g.GenreId
GROUP BY g.Name
HAVING COUNT(*) > 50
ORDER BY avg_minutes DESC
""")

,genre,tracks,avg_minutes
0,Drama,64,42.92
1,TV Shows,93,35.75
2,Metal,374,5.16
3,Classical,74,4.90
4,Jazz,130,4.86
5,Rock,1297,4.73
6,Blues,81,4.51
7,Reggae,58,4.12
8,Alternative & Punk,332,3.91
9,Latin,579,3.88


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

با `ROW_NUMBER()`، برای هر مشتری **اولین** فاکتورش را پیدا کنید. در هر ماه چند مشتری جدید داشته‌ایم؟

</div>

In [5]:
sql("""
WITH ranked AS (
    SELECT CustomerId, InvoiceDate,
           ROW_NUMBER() OVER (PARTITION BY CustomerId ORDER BY InvoiceDate) AS rn
    FROM Invoice
)
SELECT strftime('%Y-%m', InvoiceDate) AS month, COUNT(*) AS new_customers
FROM ranked
WHERE rn = 1
GROUP BY month
ORDER BY month
""")

,month,new_customers
0,2021-01,6
1,2021-02,6
2,2021-03,6
3,2021-04,5
4,2021-05,4
5,2021-06,4
6,2021-07,4
7,2021-08,3
8,2021-09,2
9,2021-10,2


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

کدام آهنگ‌ها در هیچ Playlistی نیستند؟ (راهنما: `LEFT JOIN` با `PlaylistTrack` و `IS NULL`)

</div>

In [6]:
sql("""
SELECT COUNT(*) AS tracks_in_no_playlist
FROM Track AS t
LEFT JOIN PlaylistTrack AS pt ON t.TrackId = pt.TrackId
WHERE pt.PlaylistId IS NULL
""")

,tracks_in_no_playlist
0,0


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نتیجه صفر است: در این پایگاه داده همهٔ آهنگ‌ها حداقل در یک Playlist قرار دارند. اگر به‌جای شمارش، خود سطرها را با `SELECT t.TrackId, t.Name` بگیرید، جدول خالی برمی‌گردد. الگوی `LEFT JOIN ... IS NULL` همچنان روش درست برای این سؤال است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۶

روی پایگاه دادهٔ فروشگاه (`shop`)، با تابع پنجره‌ای `LAG`، رشد ماهانهٔ فروش را حساب کنید.

</div>

In [7]:
shop = sqlite3.connect(":memory:")
pd.read_csv(DATA_URL + "orders_clean.csv").to_sql("orders", shop, index=False)

pd.read_sql_query("""
WITH monthly AS (
    SELECT strftime('%Y-%m', order_date) AS month, SUM(total_price) AS revenue
    FROM orders
    GROUP BY month
)
SELECT month,
       ROUND(revenue / 1e6, 1) AS revenue_million,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
                   / LAG(revenue) OVER (ORDER BY month), 1) AS growth_pct
FROM monthly
ORDER BY month
""", shop)

,month,revenue_million,growth_pct
0,2024-03,79.3,NaN
1,2024-04,269.0,239.2
2,2024-05,228.4,-15.1
3,2024-06,235.0,2.9
4,2024-07,277.2,17.9
5,2024-08,296.8,7.1
6,2024-09,244.9,-17.5
7,2024-10,288.4,17.7
8,2024-11,288.6,0.1
9,2024-12,218.5,-24.3
